# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).** 

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`. 

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [3]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()
    
    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))
        
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

Rozpoczynam pobieranie SEKWENCYJNE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Zaczarowana zagroda
2. Pogo
3. Życie jest piękne! Twój własny ślad
4. 19. Festiwal Muzyki Filmowej w Krakowie
5. Gerdan Theatre
6. Juwenalia Krakoskie 2026
7. Magiczna rana (Teatr im. J. Słowackiego)
8. Tańcowały dwa Michały
9. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”
10. Green ZOO Festival 2026

Czas wykonania: 4.26s


In [4]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]
    
    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        results = list(executor.map(download_site, sites))
    
    all_titles = [title for sublist in results for title in sublist]
    
    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")
        
    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...
Pobrano łącznie 100 tytułów.
Pierwsze 10 wyników:
1. Zaczarowana zagroda
2. Pogo
3. Życie jest piękne! Twój własny ślad
4. 19. Festiwal Muzyki Filmowej w Krakowie
5. Gerdan Theatre
6. Juwenalia Krakoskie 2026
7. Magiczna rana (Teatr im. J. Słowackiego)
8. Tańcowały dwa Michały
9. XXXII Międzynarodowy Festiwal „Starzy i Młodzi, czyli Jazz w Krakowie”
10. Green ZOO Festival 2026

Czas wykonania (wątki): 1.26s


--- 
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [5]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))
    
print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

Saldo końcowe: 100 zł (oczekiwano: 100)


--- 
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [6]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()
    
    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)
    
    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

Praca na 4 procesach (rdzeniach)...
Zakończono w czasie 1.07s.


---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [10]:
# Miejsce na rozwiązanie zadania 1
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"

# Twój kod tutaj...
NUM_FACTS = 20

def fetch_cat_fact(_):
    """Pobiera jeden fakt o kocie."""
    return requests.get(CAT_API_URL).json().get('fact')

# Wersja sekwencyjna
print("=== SEKWENCYJNIE ===")
start = time.time()

facts_seq = [fetch_cat_fact(None) for _ in range(NUM_FACTS)]

seq_time = time.time() - start
print(f"Pobrano {len(facts_seq)} faktów w {seq_time:.2f}s")
for i, fact in enumerate(facts_seq[:3], 1):
    print(f"{i}. {fact[:80]}...")

# Wersja wielowątkowa
print("\n=== WIELOWĄTKOWO ===")
start = time.time()

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    facts_threaded = list(executor.map(fetch_cat_fact, range(NUM_FACTS)))

thread_time = time.time() - start
print(f"Pobrano {len(facts_threaded)} faktów w {thread_time:.2f}s")
for i, fact in enumerate(facts_threaded[:3], 1):
    print(f"{i}. {fact[:80]}...")

print(f"\nPrzyspieszenie: {seq_time / thread_time:.1f}x")

=== SEKWENCYJNIE ===
Pobrano 20 faktów w 10.63s
1. Tests done by the Behavioral Department of the Musuem of Natural History conclud...
2. Some Siamese cats appear cross-eyed because the nerves from the left side of the...
3. Cats sleep 16 to 18 hours per day. When cats are asleep, they are still alert to...

=== WIELOWĄTKOWO ===
Pobrano 20 faktów w 2.57s
1. Contrary to popular belief, the cat is a social animal. A pet cat will respond a...
2. There are more than 500 million domestic cats in the world, with approximately 4...
3. Cats have 30 vertebrae (humans have 33 vertebrae during early development; 26 af...

Przyspieszenie: 4.1x


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [8]:
# Miejsce na rozwiązanie zadania 2
import queue
import threading
import time

# Twój kod tutaj...
NUM_NUMBERS = 20 
q = queue.Queue()

even_results = []
odd_results = []

def producer():
    """Generuje liczby naturalne i wrzuca je do kolejki."""
    for i in range(1, NUM_NUMBERS + 1):
        q.put(i)
        print(f"[Producent] Wyprodukowano: {i}")
        time.sleep(0.05)
    # Sentinel values – sygnał końca dla obu konsumentów
    q.put(None)
    q.put(None)

def consumer_even():
    """Pobiera i przetwarza tylko liczby parzyste."""
    while True:
        num = q.get()
        if num is None:
            q.task_done()
            break
        if num % 2 == 0:
            even_results.append(num)
            print(f"  [Konsument PARZYSTE] Przetworzono: {num}")
        else:
            # Nieparzysta – oddaj z powrotem do kolejki
            q.put(num)
        q.task_done()

def consumer_odd():
    """Pobiera i przetwarza tylko liczby nieparzyste."""
    while True:
        num = q.get()
        if num is None:
            q.task_done()
            break
        if num % 2 != 0:
            odd_results.append(num)
            print(f"  [Konsument NIEPARZYSTE] Przetworzono: {num}")
        else:
            q.put(num)
        q.task_done()

t_producer = threading.Thread(target=producer)
t_even     = threading.Thread(target=consumer_even)
t_odd      = threading.Thread(target=consumer_odd)

t_producer.start()
t_even.start()
t_odd.start()

t_producer.join()
t_even.join()
t_odd.join()

print(f"\nParzyste ({len(even_results)}):    {sorted(even_results)}")
print(f"Nieparzyste ({len(odd_results)}): {sorted(odd_results)}")

[Producent] Wyprodukowano: 1
  [Konsument NIEPARZYSTE] Przetworzono: 1
[Producent] Wyprodukowano: 2
  [Konsument PARZYSTE] Przetworzono: 2
[Producent] Wyprodukowano: 3
  [Konsument NIEPARZYSTE] Przetworzono: 3
[Producent] Wyprodukowano: 4
  [Konsument PARZYSTE] Przetworzono: 4
[Producent] Wyprodukowano: 5
  [Konsument NIEPARZYSTE] Przetworzono: 5
[Producent] Wyprodukowano: 6
  [Konsument PARZYSTE] Przetworzono: 6
[Producent] Wyprodukowano: 7
  [Konsument NIEPARZYSTE] Przetworzono: 7
[Producent] Wyprodukowano: 8
  [Konsument PARZYSTE] Przetworzono: 8
[Producent] Wyprodukowano: 9
  [Konsument NIEPARZYSTE] Przetworzono: 9
[Producent] Wyprodukowano: 10  [Konsument PARZYSTE] Przetworzono: 10

[Producent] Wyprodukowano: 11
  [Konsument NIEPARZYSTE] Przetworzono: 11
[Producent] Wyprodukowano: 12
  [Konsument PARZYSTE] Przetworzono: 12
[Producent] Wyprodukowano: 13
  [Konsument NIEPARZYSTE] Przetworzono: 13
[Producent] Wyprodukowano: 14
  [Konsument PARZYSTE] Przetworzono: 14
[Producent] Wypro

### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [11]:
# Miejsce na rozwiązanie zadania 3
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    # Twój kod tutaj...


    numbers = list(range(1, 10_001))   
    cores = multiprocessing.cpu_count()
    print(f"Liczba rdzeni: {cores}")

    # Wersja sekwencyjna dla porównania
    start = time.time()
    results_seq = [calculate_power_sum(n) for n in numbers]
    seq_time = time.time() - start
    print(f"Sekwencyjnie: {seq_time:.2f}s")

    # Wersja wieloprocesowa
    start = time.time()
    with multiprocessing.Pool(processes=cores) as pool:
        results_mp = pool.map(calculate_power_sum, numbers)
    mp_time = time.time() - start
    print(f"Wieloprocesowo: {mp_time:.2f}s")

    print(f"Przyspieszenie: {seq_time / mp_time:.1f}x")
    print(f"Przykładowe wyniki: n=1 → {results_mp[0]}, n=10 → {results_mp[9]}")

Liczba rdzeni: 4
Sekwencyjnie: 0.51s
Wieloprocesowo: 0.39s
Przyspieszenie: 1.3x
Przykładowe wyniki: n=1 → 100, n=10 → 11111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111110
